## Housing Prices in the 50 US States
The goal of this project is to clean data, find median costs of houses sold in the US via brokerages, plot the results in graphs to show the difference in states, and to save the new dataset as a CSV to use it in other programs.

In [ ]:
#import the tools we need.
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import os
os.getcwd()

In [ ]:
realtor_data = pd.read_csv("realtor_data.csv") #import the dataset

In [ ]:
realtor_data.head() #check the dataset

In [ ]:
realtor_data["state"].unique() #checking to see each unique state

### I only want to use the 50 US states and Washington D.C. for my data and houses that have a status of sold or for_sale.
I will begin cleaning the data to only contain the states by removing the US territories, after that I will remove any houses that are not currently sold or for_sale so that we can begin looking at the data.


In [ ]:
not_wanted = ["Puerto Rico", "Virgin Islands", "Guam", "New Brunswick"] #selecting locations to remove


fifty_states = realtor_data[~realtor_data['state'].isin(not_wanted).dropna()] #removing the locations not wanted and null values

fifty_states_market = fifty_states[fifty_states['status'] != 'ready_to_build'] #removing the houses that are not currently built

fifty_states_market.head()

In [ ]:
fifty_states_market["price"].isnull().sum() #checking if there are any null values for price, we can see 245 NaN values.

In [ ]:
fifty_states_market = fifty_states_market.dropna(subset=['price']) #removing all NaN values

In [ ]:
# checking to see if there are any outliers, high and low.
fifty_states_market['price'].sort_values() # we can see that some houses are being noted as sold at $1, and the largest house is over 200 million 

In [ ]:
#Drop NA values to not ruin data and calculations
fifty_states_market.dropna(subset=['bed','bath','house_size'])

#Lets add a price per square foot column to help reduce extreme outliers
fifty_states_market['pr_per_sq_ft'] = (fifty_states_market['price'] / fifty_states_market['house_size']).round(2)

#Time to remove the outliers of the bottom and top 5% of house sizes, price, and price per square foot.


low_thresh_price = fifty_states_market['price'].quantile(0.05) 
high_thresh_price = fifty_states_market['price'].quantile(0.95)

low_thresh_hs = fifty_states_market['house_size'].quantile(0.05) 
high_thresh_hs = fifty_states_market['house_size'].quantile(0.95)

low_thresh_sqft = fifty_states_market['pr_per_sq_ft'].quantile(0.05) 
high_thresh_sqft = fifty_states_market['pr_per_sq_ft'].quantile(0.95)

#this will filter out the bottom and top 5% of houses to make the data less skewed.
fifty_states_market = fifty_states_market[(fifty_states_market['price'] > low_thresh_price) & 
                                        (fifty_states_market['price'] < high_thresh_price) &
                                        (fifty_states_market['house_size'] > low_thresh_hs) &
                                        (fifty_states_market['house_size'] < high_thresh_hs) &
                                        (fifty_states_market['pr_per_sq_ft'] > low_thresh_sqft) &
                                        (fifty_states_market['pr_per_sq_ft'] < high_thresh_sqft)].sort_values(by='price') 






print(fifty_states_market)



## Next we're going to find the median values of the price, size, and price per square foot for all states and all cities.

### this data we will save and use for our future housing projects

In [ ]:
fifty_median_states = fifty_states_market.groupby('state')[['price', 'bed', 'bath', 'house_size', 'pr_per_sq_ft']].agg(['median']).round()
fifty_median_cities = fifty_states_market.groupby(['state','city'])[['price', 'bed', 'bath', 'house_size', 'pr_per_sq_ft']].agg('median',).round()
print(fifty_median_states)

In [ ]:
fifty_median_states['price'].plot(kind='bar', title='Median House Price per State', xlabel='States', ylabel='Price', rot=90, width=0.25)

In [ ]:
plt.scatter(fifty_median_states.index, fifty_median_states['pr_per_sq_ft'])

In [ ]:
#saving the DFs for later use.
fifty_median_states.to_csv('fifty_median_states.csv', index=False)
fifty_median_cities.to_csv('fifty_median_cities.csv', index=False)